<a href="https://colab.research.google.com/github/vinol3/WB2_lab1/blob/main/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformers datasets huggingface_hub accelerate

In [ ]:
%%python --version

In [ ]:
%pip install torch torchvision transformers datasets huggingface_hub

In [ ]:
import torch

# This tells PyTorch to use the GPU if it's available, otherwise fallback to CPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"You are currently using a: {device.upper()}")

In [ ]:
import torchvision.models as models

# 1. Load the ResNet-18 model. "weights='DEFAULT'" loads a pre-trained version.
resnet = models.resnet18(weights='DEFAULT')
print(type(resnet))

# 2. Move the model to our device (GPU)
resnet = resnet.to(device)
resnet.eval()  # Set the model to evaluation mode (important for inference)

print("ResNet successfully loaded and moved to:", device)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

# We use streaming=True to avoid downloading 100+ GB of data!
print("Connecting to ImageNet-1k...")
dataset = load_dataset("imagenet-1k", split="train", streaming=True, trust_remote_code=True)
print(type(dataset))

# Look at the very first image in the dataset
iterator = iter(dataset)
first_sample = next(iterator)

print("\nDataset Labels:", first_sample['label'])
first_sample['image']

In [ ]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests

# 1. Load the model and the processor (which prepares our images/text for the model)
model_id = "openai/clip-vit-base-patch32"
### STUDENT CODE HERE ###
clip_model.to(device)

# 2. Get a sample image from the internet
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# 3. Give the model some text options to guess what is in the image
text_options = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

# 4. Process the inputs and run the model!
inputs = processor(text=text_options, images=image, return_tensors="pt", padding=True)
inputs = inputs.to(device)

outputs = clip_model(**inputs)

# 5. Look at the results
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)

print("Probability it is a cat:", probs[0][0].item())
print("Probability it is a dog:", probs[0][1].item())
print("Probability it is a car:", probs[0][2].item())

In [ ]:
from torchvision import transforms

# ResNet requires images to be resized and normalized in a very specific way
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Process the image and move it to the GPU
input_tensor = preprocess(image).unsqueeze(0)
print(input_tensor.shape)  # Should be [1, 3, 224, 224]

# Run the model
with torch.inference_mode():
    model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
    output = model(input_tensor)
print(output.shape) # Should be [1, 1000]

# Get the highest predicted score
predicted_class = output.argmax(dim=1).item()
print(f"ResNet predicts this is class ID: {predicted_class}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [ ]:
response

In [ ]:
# pip install accelerate
import requests
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", device_map="auto")

img_url = 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg'
raw_image = Image.open(requests.get(img_url, stream=True).raw).convert('RGB')

question = "how many dogs are in the picture?"
inputs = processor(raw_image, question, return_tensors="pt").to("cuda")

out = model.generate(**inputs)
print(processor.decode(out[0], skip_special_tokens=True).strip())
